# AI Cybersecurity Demo 1: Phishing URL Classifier

**Goal:** Paste a URL and use a pretrained Hugging Face model to classify it as likely legitimate or likely phishing, display the model's confidence score, and show some simple URL warning signs.

**Model:** [`CrabInHoney/urlbert-tiny-v4-phishing-classifier`](https://huggingface.co/CrabInHoney/urlbert-tiny-v4-phishing-classifier)

## What this demonstrates
- AI as a cybersecurity defender: automated phishing URL detection.
- Text/URL classification with a pretrained transformer model.
- Why confidence is not the same as certainty.

> **Safety and limitations:** This demo analyzes the URL text only. It does not visit or open the website. Use example or otherwise safe URLs; do not click suspicious links. Model output can be wrong, so never treat this demo as a security guarantee or as the sole basis for blocking a website.

### Easiest way to run
Upload this notebook to **Google Colab** or open it in Jupyter. Run the cells from top to bottom. The first model load requires internet access to download the pretrained model.

## 1. Install the required packages

Run this cell once in the current notebook environment. In Google Colab, PyTorch may already be installed.

In [ ]:
%pip -q install transformers torch matplotlib

## 2. Load the pretrained model

The model card describes `LABEL_0` as good/non-phishing and `LABEL_1` as phishing. The code below uses that mapping and prints the model's configured labels for transparency.

In [ ]:
import re
import ipaddress
from urllib.parse import urlsplit

import matplotlib.pyplot as plt
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

MODEL_NAME = "CrabInHoney/urlbert-tiny-v4-phishing-classifier"
DEVICE = 0 if torch.cuda.is_available() else -1

print("Loading model; the first download may take a little while...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=DEVICE,
)
print("Model ready!")
print("Model labels:", model.config.id2label)
print("Running on:", "GPU" if DEVICE >= 0 else "CPU")

## 3. Define the URL checker

The model provides the prediction and score. The warning-sign checks are simple teaching heuristics shown separately; **they are not a faithful explanation of the model's internal reasoning**.

In [ ]:
def normalize_url(raw_url: str) -> str:
    """Trim input and add https:// when a user enters only a domain/path."""
    url = raw_url.strip()
    if not url:
        raise ValueError("Please enter a URL or domain name.")
    if not re.match(r"^https?://", url, flags=re.IGNORECASE):
        url = "https://" + url
    return url


def human_label(label: str) -> str:
    """Translate the Hugging Face model's output label to a readable verdict."""
    label_upper = str(label).upper()
    label_text = str(label).lower()
    if label_upper in {"LABEL_1", "1"} or "phish" in label_text or "malicious" in label_text:
        return "LIKELY PHISHING / MALICIOUS"
    if label_upper in {"LABEL_0", "0"} or "good" in label_text or "benign" in label_text or "legitimate" in label_text:
        return "LIKELY LEGITIMATE"
    return f"MODEL LABEL: {label}"


def url_warning_signs(url: str):
    """Return basic, transparent URL signals for classroom discussion."""
    parsed = urlsplit(url)
    host = parsed.hostname or ""
    flags = []

    if len(url) > 75:
        flags.append("Long URL (over 75 characters)")
    if "@" in url:
        flags.append("Contains '@' (can obscure the apparent destination)")
    try:
        ipaddress.ip_address(host)
        flags.append("Uses an IP address instead of a domain name")
    except ValueError:
        pass
    if "xn--" in host.lower():
        flags.append("Contains punycode (internationalized domain encoding)")
    if host.count("-") >= 2:
        flags.append("Several hyphens in the hostname")
    if len(host.split(".")) >= 4:
        flags.append("Many hostname labels/subdomains")
    if re.search(r"(login|verify|password|account|billing|update|secure)", url, re.I):
        flags.append("Contains a word sometimes used in credential-themed links")
    if parsed.scheme.lower() != "https":
        flags.append("Does not use HTTPS (weak signal only; HTTPS alone does not prove safety)")

    return flags


def classify_url(raw_url: str, show_plot: bool = True):
    url = normalize_url(raw_url)

    # Request all class scores, rather than only the top class.
    raw_scores = classifier(url, top_k=None)
    if raw_scores and isinstance(raw_scores[0], list):
        raw_scores = raw_scores[0]

    scores = {item["label"]: float(item["score"]) for item in raw_scores}
    top = max(raw_scores, key=lambda item: float(item["score"]))
    verdict = human_label(top["label"])
    confidence = float(top["score"])

    print("=" * 68)
    print("URL checked:", url)
    print("AI verdict:", verdict)
    print(f"Model confidence score: {confidence:.1%}")
    print("\nScores for each model class:")
    for label, score in sorted(scores.items(), key=lambda pair: pair[1], reverse=True):
        print(f"  {human_label(label):30s} {score:.1%}  ({label})")

    flags = url_warning_signs(url)
    print("\nSimple URL warning-sign checks (separate from model explanation):")
    if flags:
        for flag in flags:
            print("  ⚠", flag)
    else:
        print("  No warning signs detected by these simple rules.")
    print("\nReminder: a clean-looking URL can still be malicious, and a flagged URL is not automatically malicious.")
    print("=" * 68)

    if show_plot:
        display_labels = [human_label(label).replace("LIKELY ", "") for label in scores]
        values = [scores[label] * 100 for label in scores]
        plt.figure(figsize=(7, 3.5))
        bars = plt.barh(display_labels, values)
        plt.xlim(0, 100)
        plt.xlabel("Model score (%)")
        plt.title("Phishing URL classifier: class scores")
        plt.gca().invert_yaxis()
        for bar, value in zip(bars, values):
            plt.text(min(value + 1, 88), bar.get_y() + bar.get_height()/2, f"{value:.1f}%", va="center")
        plt.tight_layout()
        plt.show()

    return {"url": url, "verdict": verdict, "confidence": confidence, "scores": scores, "warning_signs": flags}

## 4. Try two example URLs

These examples are based on the model card's own examples. A result is an output of this pretrained model, not an independent safety verdict.

In [ ]:
example_urls = [
    "https://huggingface.co/",
    "http://hu991ngface.com.ru/",
]

for example in example_urls:
    classify_url(example, show_plot=True)

## 5. Live demo: paste a URL

Run the next cell and paste a URL into the prompt. For the classroom, keep these examples handy:

- `https://huggingface.co/`
- `http://hu991ngface.com.ru/`
- A URL from a safe, documented phishing-training dataset

Do not use the notebook to open the address. It only sends the URL string to the model for classification.

In [ ]:
try:
    user_url = input("Paste a URL to classify (or press Enter to skip): ").strip()
    if user_url:
        result = classify_url(user_url, show_plot=True)
    else:
        print("Skipped live URL input.")
except ValueError as exc:
    print(exc)

## 6. How to explain the demo to the class

1. **Input:** The analyst supplies a URL as text; the notebook does not visit the site.
2. **AI processing:** A pretrained transformer classifier evaluates the URL representation it learned during training.
3. **Output:** The model returns class scores. We display the top class and its score as a confidence-style measure.
4. **Interpretation:** The simple warning-sign list is a separate, transparent checklist—not SHAP and not proof of what the model used internally.
5. **Limitations:** Scores can be wrong or poorly calibrated. Attackers change URLs, and models can fail on unfamiliar patterns. A production system needs representative testing, monitoring, and other security controls.

### Suggested 30-second explanation

“This demo uses a pretrained AI model to classify a URL as likely phishing or likely legitimate. It returns scores for both classes. I also show some basic URL warning signs, but those rules are separate from the AI model. The result helps an analyst make a decision; it does not guarantee that a website is safe.”

## Source

- Model and model-card usage example: [CrabInHoney/urlbert-tiny-v4-phishing-classifier](https://huggingface.co/CrabInHoney/urlbert-tiny-v4-phishing-classifier)
- The model card identifies `LABEL_0` as good and `LABEL_1` as phishing, and reports evaluation metrics for its own testing setup. Those metrics should not be interpreted as a guarantee of real-world performance.